In [11]:
import h5py,os,re,click
import numpy as np

ind=3
ens=['cB211.072.64','cC211.060.80','cD211.054.96','cE211.044.112'][ind]
label=['B64','C80','D96','E112'][ind]

t='cfgs_charges' if ens in ['cE211.044.112'] else 'cfgs_run' 
path=f'/p/project1/ngff/li47/code/glwc3/project2/05_moments/dataPrepare/{ens}/data_aux/{t}'
with open(path,'r') as f:
    cfgs=f.read().splitlines()
cfg2old=lambda cfg: cfg[1:]+'_r'+{'a':'0','b':'1','c':'2','d':'3'}[cfg[0]]
cfg2new=lambda cfg: {'0':'a','1':'b','2':'c','3':'d'}[cfg[-1]] + cfg[:4]

path=f'/p/project1/ngff/li47/code/projectData/07_Nsgm/charges/{ens}.h5'
with h5py.File(path,'w') as fw:
    fw.create_dataset('cfgs',data=cfgs)
    
    path=f'/p/project1/ngff/li47/code/projectData/07_Nsgm/fromChristos/charges_{label}.h5'
    with h5py.File(path) as f:
        for key in f.keys():
            if not key.endswith('_twop'):
                continue
            tf=int(key.split('_')[0][2:])
            t=np.array([f[key][cfg2old(cfg)][()] for cfg in cfgs])
            t=t[:,None][:,[0]*(tf+1)]
            fw.create_dataset(f'conn_c2pt/{tf}',data=t)
        
        for g in ['gS','gA','gT']:
            for tf_str in f[f'{g}/up'].keys():
                tf=int(tf_str[2:])
                tu=np.array([f[f'{g}/up/{tf_str}/{cfg2old(cfg)}'][:] for cfg in cfgs])
                td=np.array([f[f'{g}/dn/{tf_str}/{cfg2old(cfg)}'][:] for cfg in cfgs])
                
                factor=-1 if label in ['D96','E112'] else +1
                
                fw.create_dataset(f'conn_{g}+/{tf}',data=(tu+td)*factor)
                fw.create_dataset(f'conn_{g}-/{tf}',data=tu-td)
                
    # path=f'/p/project1/ngff/li47/code/projectData/05_moments/{ens}/data_merge/conn_2pt.h5'
    # with h5py.File(path) as f:
    #     moms=[list(mom) for mom in f['moms'][:]]
    #     ind=moms.index([0,0,0])
    #     for tf in f['data'].keys():
    #         t=f['data'][tf][:,:,ind]
    #         tf=int(tf)
    #         fw.create_dataset(f'conn_c2pt/{tf}',data=t)
    
    t='disc_2pt_charges.h5' if ens in ['cE211.044.112'] else 'disc_2pt.h5'
    path=f'/p/project1/ngff/li47/code/projectData/05_moments/{ens}/data_merge/{t}'
    with h5py.File(path) as f:
        t=[cfg.decode() for cfg in f['cfgs'][:]]
        dic={}
        for i,cfg in enumerate(t):
            dic[cfg]=i
        inds_cfg=[dic[cfg] for cfg in cfgs]
        
        moms=[list(mom) for mom in f['moms'][:]]
        ind=moms.index([0,0,0])
        t=f['data/N_N'][:,:,ind]
        fw.create_dataset(f'disc_c2pt',data=t[inds_cfg])
        
    path=f'/p/project1/ngff/li47/code/scratch/run/05_moments_run5_local/{ens}/data_merge/disc_0,0,0,0,0,0.h5'
    with h5py.File(path) as f:
        t=[cfg.decode() for cfg in f['cfgs'][:]]
        dic={}
        for i,cfg in enumerate(t):
            dic[cfg]=i
        inds_cfg=[dic[cfg] for cfg in cfgs]
        
        moms=[list(mom) for mom in f['moms'][:]]
        ind=moms.index([0,0,0,0,0,0])
        
        projs=['P0', 'Px', 'Py', 'Pz']
        inserts=[insert.decode() for insert in f['inserts'][:]]
        
        for g,proj,gm,factor in zip(['gS','gA','gT'],['P0','Pz','Pz'],['id','g5gz','sgmxy'],[1,1j,-1j]):
            for key in f['data'].keys():
                j,tf=key.split('_')
                fla=j[1]; tf=int(tf)
                
                t=f[f'data/{key}'][:]
                t=t[:,:,0,projs.index(proj),inserts.index(gm)]
                t=np.real(t*factor)
                
                fw.create_dataset(f'disc_{g}{fla}/{tf}',data=t[inds_cfg])
                
    path=f'/p/project1/ngff/li47/code/projectData/05_moments/{ens}/data_merge/disc_jvev_local.h5'
    with h5py.File(path) as f:
        # t=[cfg.decode() for cfg in f['cfgs'][:]]
        # dic={}
        # for i,cfg in enumerate(t):
        #     dic[cfg]=i
        # inds_cfg=[dic[cfg] for cfg in cfgs]
        
        for key in f.keys():
            fla=key[1]
            fw.create_dataset(f'vev_gS{fla}',data=f[key][inds_cfg,0])

In [17]:
import h5py

def merge_hdf5(file1, file2, output):
    with h5py.File(output,'w') as fw, h5py.File(file1) as f1, h5py.File(file2) as f2:
        for key in f1.keys():
            fw.copy(f1[key],fw)
        
        def copy_recursive(src, dest):
            for key, item in src.items():
                if key not in dest:
                    src.copy(key, dest)
                    continue

                if isinstance(item, h5py.Group) and isinstance(dest[key], h5py.Group):
                    copy_recursive(item, dest[key])
        
        copy_recursive(f2,fw)
            
p1='/p/project1/ngff/li47/code/projectData/07_Nsgm/fromChristos/charges_8_to_20_D96.h5'
p2='/p/project1/ngff/li47/code/projectData/07_Nsgm/fromChristos/charges_22_to_26_D96.h5'
p3='/p/project1/ngff/li47/code/projectData/07_Nsgm/fromChristos/charges_D96.h5'
merge_hdf5(p1,p2,p3)